# PRAGma App — LOT 기반 에칭 공정 분석 시스템

**파이프라인:**
1. LOT 번호 입력 → MES(CSV Mock)에서 공정 데이터 자동 조회
2. LightGBM 모델로 에칭 속도 예측
3. **RAG 기반 LLM 분석** (Hybrid BM25+Vector + ML 예측/OPLS 컨텍스트 주입)으로 공정 해석 및 조치 방안 제시
   - `USE_GEMINI = True` → **Gemini API** (gemini-2.5-flash + gemini-embedding-001) — 모델 다운로드 없이 빠르게 구동
   - `USE_GEMINI = False` → **로컬 EXAONE-3.5-7.8B** (HuggingFace, GPU 4bit 양자화)

**환경 설정 셀의 `USE_GEMINI` 플래그로 두 백엔드를 전환할 수 있습니다.**
- Gemini 사용 시 → **Secrets에 `GEMINI_KEY` 필요** (Google AI Studio에서 발급, GPU 불필요)
- 로컬 EXAONE 사용 시 → **Secrets에 `HF_TOKEN` 필요**, **Google Colab GPU 런타임 필수** (4bit 양자화)

## 0. 환경 설정

In [102]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. GitHub에서 코드 가져오기
import subprocess
result = subprocess.run(["git", "clone", "https://github.com/heyitsmialee/PRAGma.git"],
                        capture_output=True, text=True)
if "already exists" in result.stderr:
    print("PRAGma 이미 존재 — pull로 최신화")
    subprocess.run(["git", "-C", "/content/PRAGma", "pull"], check=True)
else:
    print(result.stdout or result.stderr)

import os
os.chdir("/content/PRAGma")
print(f"작업 디렉토리: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PRAGma 이미 존재 — pull로 최신화
작업 디렉토리: /content/PRAGma


In [103]:
import os
import json
import pickle
import subprocess
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ── 모델 백엔드 선택 ──────────────────────────────────────────────────────────
# True  → Gemini API (gemini-2.5-flash + gemini-embedding-001, 모델 다운로드 불필요)
# False → 로컬 EXAONE-3.5-7.8B (HuggingFace, GPU 4bit 양자화 필요)
USE_GEMINI = True
os.environ["USE_GEMINI"] = "true" if USE_GEMINI else "false"

# ── 패키지 설치 ───────────────────────────────────────────────────────────────
PACKAGES = [
    "langchain", "langchain-core", "langchain-community",
    "langchain-text-splitters", "langchain-chroma", "rank_bm25",
    "streamlit", "plotly", "shap",
]
if USE_GEMINI:
    PACKAGES.append("langchain-google-genai")
else:
    PACKAGES.append("langchain-huggingface")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + PACKAGES, check=True)
if not USE_GEMINI:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.52.0", "accelerate", "bitsandbytes", "torch"], check=True)
print("패키지 설치 완료")

# ── 인증 정보 (Colab Secrets) ─────────────────────────────────────────────────
from google.colab import userdata

if USE_GEMINI:
    GEMINI_API_KEY = userdata.get("GEMINI_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("Secrets에 GEMINI_KEY가 없습니다. 왼쪽 🔑 → Add new secret 으로 추가하세요.")
    # Streamlit 서브프로세스가 환경변수를 상속받도록 설정
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("Gemini API 키 설정 완료")
else:
    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        raise ValueError("Secrets에 HF_TOKEN이 없습니다. 왼쪽 🔑 → Add new secret 으로 추가하세요.")
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HuggingFace 로그인 완료")

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
# GitHub clone 경로 우선, Drive fallback
GITHUB_DIR = Path("/content/PRAGma")
DRIVE_DIR  = Path("/content/drive/MyDrive/Colab Notebooks/PRAGma")

MODEL_PATH      = GITHUB_DIR / "notebooks/best_LightGBM_mass_speed_regressor.pkl"
CSV_PATH        = GITHUB_DIR / "data/Train_0319.csv"
PAPER_JSON_PATH = GITHUB_DIR / "data/rag_data_all.json"
SHAP_MD_PATH    = GITHUB_DIR / "notebooks/shap_analysis_for_rag.md"
CHROMA_DIR      = "/content/pragma_chroma"

# Drive fallback (GitHub에 없는 경우)
if not MODEL_PATH.exists():
    MODEL_PATH = DRIVE_DIR / "best_LightGBM_mass_speed_regressor.pkl"
if not CSV_PATH.exists():
    CSV_PATH   = DRIVE_DIR / "Train_0319.csv"

print(f"모델 경로  : {MODEL_PATH}")
print(f"CSV 경로   : {CSV_PATH}")
print(f"JSON 경로  : {PAPER_JSON_PATH}")
print(f"SHAP MD    : {SHAP_MD_PATH}")
print("설정 완료")

패키지 설치 완료
Gemini API 키 설정 완료
모델 경로  : /content/PRAGma/notebooks/best_LightGBM_mass_speed_regressor.pkl
CSV 경로   : /content/PRAGma/data/Train_0319.csv
JSON 경로  : /content/PRAGma/data/rag_data_all.json
SHAP MD    : /content/PRAGma/notebooks/shap_analysis_for_rag.md
설정 완료


## 1. 모델 및 데이터 로드

In [104]:
# LightGBM 모델 로드
with open(MODEL_PATH, "rb") as f:
    lgbm_model = pickle.load(f)

MODEL_FEATURES = lgbm_model.feature_name_
print(f"모델 로드 완료 — 피처 수: {len(MODEL_FEATURES)}")
print(f"피처 목록: {MODEL_FEATURES[:8]} ...")

# MES Mock 데이터 로드 (실제 환경에서는 MES API 호출로 대체)
df_mes = pd.read_csv(CSV_PATH, encoding="cp949")
print(f"\nMES Mock 데이터 로드 완료 — LOT 수: {len(df_mes)}, 컬럼 수: {len(df_mes.columns)}")
print(f"LOT 예시: {df_mes['LOT'].head(5).tolist()}")

# RAG 지식베이스 로드
with open(PAPER_JSON_PATH, "r", encoding="utf-8") as f:
    paper_rules = json.load(f)

shap_content = SHAP_MD_PATH.read_text(encoding="utf-8") if SHAP_MD_PATH.exists() else ""

print(f"\nRAG 지식베이스 로드 완료")
print(f"  논문 rule 수: {len(paper_rules)}")
print(f"  SHAP 분석 MD 존재: {bool(shap_content)}")

모델 로드 완료 — 피처 수: 45
피처 목록: ['cu_thick_max', 'cu_thick_avg', 'cu_thick_min', 'cu_thick_std', 'cu_thick_median', 'etch_factor', 'meas_etch_cu', 'meas_etch_hcl'] ...

MES Mock 데이터 로드 완료 — LOT 수: 8166, 컬럼 수: 103
LOT 예시: ['A20000', 'A20001', 'A20002', 'A20003', 'A20004']

RAG 지식베이스 로드 완료
  논문 rule 수: 5
  SHAP 분석 MD 존재: False


## 2. CSV 컬럼 → 모델 피처 매핑

실제 MES에서 받는 raw 컬럼명을 LightGBM 모델 입력 피처명으로 변환합니다.

In [105]:
# ── 연속형 컬럼 매핑 (동적 탐색) ─────────────────────────────────────────────
CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val"   : "cu_thick_max",
    "Cu 표면두께 AVG_VAL"   : "cu_thick_avg",
    "Cu 표면두께 Min_Val"   : "cu_thick_min",
    "Cu 표면두께 Std_Val"   : "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}

_feat_suffix_map = {
    "Etch factor"             : "etch_factor",
    "Etching(염화동) - Cu"    : "meas_etch_cu",
    "Etching(염화동) - HCl"   : "meas_etch_hcl",
    "Etching(염화동) - 비중"  : "meas_etch_sg",
    "Etching(염화동) - 온도"  : "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량"               : "meas_etch_amount",
    "Soft Etch - Cu"          : "meas_softetch_cu",
    "Soft Etch - H2SO4"       : "meas_softetch_h2so4",
    "Soft Etch - SPS"         : "meas_softetch_sps",
    "박리액 - 농도"           : "meas_strip_conc",
    "수세수 - pH"             : "meas_rinse_ph",
    "현상액 - pH"             : "meas_dev_ph",
    "현상액 - 농도"           : "meas_dev_conc",
}

for col in df_mes.columns:
    if "분석치" in col:
        suffix = col.split("_", 1)[-1] if "_" in col else col
        if suffix in _feat_suffix_map:
            CONTINUOUS_MAP[col] = _feat_suffix_map[suffix]

CONTINUOUS_MAP_RESOLVED = CONTINUOUS_MAP

print(f"매핑 완료: {len(CONTINUOUS_MAP_RESOLVED)}개 연속형 피처")
for k, v in CONTINUOUS_MAP_RESOLVED.items():
    print(f"  {k!r:45s} → {v}")

매핑 완료: 19개 연속형 피처
  'Cu 표면두께 Max_Val'                             → cu_thick_max
  'Cu 표면두께 AVG_VAL'                             → cu_thick_avg
  'Cu 표면두께 Min_Val'                             → cu_thick_min
  'Cu 표면두께 Std_Val'                             → cu_thick_std
  'Cu 표면두께 Median_Val'                          → cu_thick_median
  '분석치_Etch factor'                             → etch_factor
  '분석치_Etching(염화동) - Cu'                       → meas_etch_cu
  '분석치_Etching(염화동) - HCl'                      → meas_etch_hcl
  '분석치_Etching(염화동) - 비중'                       → meas_etch_sg
  '분석치_Etching(염화동) - 온도'                       → meas_etch_temp
  '분석치_Etching-첨가제(HB-120EF)'                   → meas_etch_additive
  '분석치_Etching량'                                → meas_etch_amount
  '분석치_Soft Etch - Cu'                          → meas_softetch_cu
  '분석치_Soft Etch - H2SO4'                       → meas_softetch_h2so4
  '분석치_Soft Etch - SPS'                         → meas_softetch_sps
  '분석치

## 3. MES Mock 함수

실제 환경에서는 MES REST API 호출로 교체합니다.

In [106]:
def get_lot_data(lotno: str) -> dict | None:
    """MES Mock: LOT 번호 → 공정 데이터 딕셔너리 반환

    실제 MES 연동 시 이 함수만 교체하면 됩니다:
        response = requests.get(f"{MES_URL}/lot/{lotno}")
        return response.json()
    """
    rows = df_mes[df_mes["LOT"] == lotno]
    if rows.empty:
        print(f"[경고] LOT '{lotno}' 를 찾을 수 없습니다.")
        print(f"  사용 가능한 LOT 예시: {df_mes['LOT'].head(10).tolist()}")
        return None
    return rows.iloc[0].to_dict()


# 테스트
sample = get_lot_data("A20000")
if sample:
    print(f"LOT A20000 조회 성공")
    print(f"  실제 에칭 속도: {sample.get('부식 Speed')} m/min")
    print(f"  에칭 온도: {sample.get('분析치_Etching(염화동) - 온도')} °C")
    print(f"  에칭 비중: {sample.get('분析치_Etching(염화동) - 비중')}")

LOT A20000 조회 성공
  실제 에칭 속도: 2.8 m/min
  에칭 온도: None °C
  에칭 비중: None


## 4. 피처 변환 (MES 데이터 → LightGBM 입력)

In [107]:
# ── 범주형 컬럼 정의 ───────────────────────────────────────────────────────────
CATEGORICAL_COLS = {
    "재작업사유" : {
        "prefix": "rework_history",
        "values": ["Unknown", "기타", "기판 겹침", "두께 미달", "딤플", "설비 에러"],
        "sep"   : "_",   # 모델은 공백을 언더스코어로 저장했음
    },
    "노광 설비정보": {
        "prefix": "expo_eq_id",
        "values": [f"EXP-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
    "DES 설비정보": {
        "prefix": "des_eq_id",
        "values": [f"DES-{i:03d}" for i in range(1, 7)],
        "sep"   : "_",
    },
    "정면 설비정보": {
        "prefix": "brush_eq_id",
        "values": [f"PRE-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
}


def prepare_features(lot_data: dict) -> pd.DataFrame:
    """MES 딕셔너리 → LightGBM 입력 DataFrame (1행)"""
    row = {}

    # 1. 연속형 피처
    for csv_col, feat_name in CONTINUOUS_MAP_RESOLVED.items():
        val = lot_data.get(csv_col, np.nan)
        row[feat_name] = float(val) if val is not None and str(val) not in ("", "nan", "None") else np.nan

    # 2. 범주형 → one-hot
    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw_val = str(lot_data.get(csv_col, "")).strip()
        # 공백을 언더스코어로 변환 (rework_history 한정)
        norm_val = raw_val.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw_val
        for v in cfg["values"]:
            norm_v = v.replace(" ", "_") if cfg["prefix"] == "rework_history" else v
            feat_key = f"{cfg['prefix']}{cfg['sep']}{norm_v}"
            row[feat_key] = 1 if norm_val == norm_v else 0

    # 3. 모델이 요구하는 피처 순서로 정렬, 없는 피처는 0 채움
    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0

    return pd.DataFrame([row])[MODEL_FEATURES]


# 테스트
if sample:
    X = prepare_features(sample)
    print(f"피처 변환 완료: shape = {X.shape}")
    print(X[[
        "cu_thick_avg", "etch_factor", "meas_etch_temp",
        "meas_etch_sg", "meas_etch_cu", "expo_eq_id_EXP-001"
    ]].to_string(index=False))

피처 변환 완료: shape = (1, 45)
 cu_thick_avg  etch_factor  meas_etch_temp  meas_etch_sg  meas_etch_cu  expo_eq_id_EXP-001
    17.919439     5.291029       48.121027      1.369145    150.468409                   1


## 5. ML 예측 (LightGBM)

In [108]:
def predict_etch_speed(lot_data: dict) -> tuple[float, pd.DataFrame]:
    """LightGBM으로 에칭 속도 예측

    Returns:
        (예측값, 피처 DataFrame)
    """
    X = prepare_features(lot_data)
    pred = lgbm_model.predict(X)[0]
    return float(pred), X


def get_opls_bounds() -> dict:
    """에칭 공정 OPLS 기준값 (하드코딩 — 실제는 MES에서 조회)"""
    return {
        "meas_etch_temp"    : {"lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C",  "name": "에칭 온도"},
        "meas_etch_sg"      : {"lcl": 1.32,  "sl": 1.37,  "ucl": 1.42,  "unit": "",    "name": "에칭 비중"},
        "meas_etch_cu"      : {"lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L", "name": "에칭 Cu 농도"},
        "meas_etch_hcl"     : {"lcl": 0.3,   "sl": 0.5,   "ucl": 0.7,   "unit": "N",   "name": "에칭 HCl"},
        "meas_etch_additive": {"lcl": 2.6,   "sl": 3.0,   "ucl": 3.4,   "unit": "g/L", "name": "에칭 첨가제"},
    }


def check_opls_status(X: pd.DataFrame) -> list[dict]:
    """OPLS 기준 대비 이탈 항목 체크"""
    bounds = get_opls_bounds()
    alerts = []
    for feat, lim in bounds.items():
        if feat not in X.columns:
            continue
        val = X[feat].iloc[0]
        if pd.isna(val):
            continue
        status = "정상"
        if val > lim["ucl"]:
            status = "UCL 초과 (상한 이탈)"
        elif val < lim["lcl"]:
            status = "LCL 미달 (하한 이탈)"
        elif val > lim["sl"] * 1.02:
            status = "SL 상향 근접"
        elif val < lim["sl"] * 0.98:
            status = "SL 하향 근접"
        alerts.append({
            "피처"  : feat,
            "항목"  : lim["name"],
            "현재값": round(val, 4),
            "LCL"   : lim["lcl"],
            "SL"    : lim["sl"],
            "UCL"   : lim["ucl"],
            "단위"  : lim["unit"],
            "상태"  : status,
        })
    return alerts


# 테스트
if sample:
    pred_speed, X = predict_etch_speed(sample)
    actual_speed  = sample.get("부식 Speed", "N/A")
    print(f"=== LOT: {sample['LOT']} ===")
    print(f"예측 에칭 속도 : {pred_speed:.4f} m/min")
    print(f"실제 에칭 속도 : {actual_speed} m/min")
    if isinstance(actual_speed, (int, float)):
        err = abs(pred_speed - actual_speed) / actual_speed * 100
        print(f"오차율          : {err:.2f}%")

    print("\n=== OPLS 상태 ===" )
    alerts = check_opls_status(X)
    df_alerts = pd.DataFrame(alerts)
    print(df_alerts.to_string(index=False))

=== LOT: A20000 ===
예측 에칭 속도 : 1.9959 m/min
실제 에칭 속도 : 2.8 m/min
오차율          : 28.72%

=== OPLS 상태 ===
                피처       항목      현재값    LCL     SL    UCL  단위       상태
    meas_etch_temp    에칭 온도  48.1210  44.50  48.00  53.00  °C       정상
      meas_etch_sg    에칭 비중   1.3691   1.32   1.37   1.42           정상
      meas_etch_cu 에칭 Cu 농도 150.4684 125.00 155.00 185.00 g/L SL 하향 근접
     meas_etch_hcl   에칭 HCl   0.5198   0.30   0.50   0.70   N SL 상향 근접
meas_etch_additive   에칭 첨가제   3.0212   2.60   3.00   3.40 g/L       정상


## 6. Streamlit 앱 코드 생성

In [135]:
from pathlib import Path

APP_CODE = r'''
import os
import json
import pickle
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
USE_GEMINI = os.environ.get("USE_GEMINI", "true").strip().lower() not in ("0", "false", "no")

if USE_GEMINI:
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
else:
    import torch
    from transformers import BitsAndBytesConfig
    from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
    from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import shap
import plotly.graph_objects as go


BASE_DIR = Path(__file__).parent
DATA_DIR = BASE_DIR / "data"
NOTEBOOK_DIR = BASE_DIR / "notebooks"
MODEL_DIR = BASE_DIR / "models"

MODEL_CANDIDATES = [
    NOTEBOOK_DIR / "best_LightGBM_mass_speed_regressor.pkl",
    MODEL_DIR / "best_LightGBM_mass_speed_regressor.pkl",
    BASE_DIR / "best_LightGBM_mass_speed_regressor.pkl",
]

SHAP_CANDIDATES = [
    NOTEBOOK_DIR / "shap_analysis_for_rag.md",
    DATA_DIR / "shap_analysis_for_rag.md",
    BASE_DIR / "shap_analysis_for_rag.md",
]

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

CSV_PATH = DATA_DIR / "Train_0319.csv"
PAPER_JSON_PATH = DATA_DIR / "rag_data_all.json"
OPLS_JSON_PATH = DATA_DIR / "opls_process_knowledge.json"


st.set_page_config(page_title="PRAGma", page_icon="⚙️", layout="wide")

st.markdown("""
<style>
.stApp {
    background-color: #ffffff;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Pretendard", "Apple SD Gothic Neo", Roboto, "Helvetica Neue", Arial, sans-serif;
}

/* 모니터링 설비 선택(셀렉트박스)과 LOT/설비 정보(비활성 입력창)의 폰트·색상 통일 */
[data-testid="stSelectbox"] div[data-baseweb="select"] > div,
div[class*="st-key-lot_info_box"] input {
    font-size: 14px !important;
    font-weight: 600 !important;
    color: #0f172a !important;
    -webkit-text-fill-color: #0f172a !important;
}

div[class*="st-key-lot_info_box"] input:disabled {
    background-color: #ffffff !important;
    opacity: 1 !important;
    cursor: default !important;
}

div[class*="st-key-lot_info_box"] label p {
    color: #64748b !important;
    font-weight: 600 !important;
    font-size: 13px !important;
}

div[class*="st-key-lot_no_input"] label p,
div[class*="st-key-monitor_eq_select"] label p {
    font-weight: 700 !important;
    font-size: 14px !important;
    color: #0f172a !important;
}

/* 전체 톤앤매너(그린 계열)에 맞춰 primary 버튼 색상 통일 */
.stApp button[kind="primary"] {
    background-color: #1e7565 !important;
    border-color: #1e7565 !important;
    color: #ffffff !important;
}

.stApp button[kind="primary"]:hover {
    background-color: #28544d !important;
    border-color: #28544d !important;
    color: #ffffff !important;
}

.stApp button[kind="primary"]:active,
.stApp button[kind="primary"]:focus:not(:hover) {
    background-color: #1e7565 !important;
    border-color: #1e7565 !important;
    color: #ffffff !important;
}

.block-container {
    max-width: 1560px;
    padding-top: 2.6rem;
    padding-bottom: 2.4rem;
    padding-left: 3.2rem;
    padding-right: 3.2rem;
}

[data-testid="stAppViewContainer"], [data-testid="stMain"], .main, section.main {
    transform: none !important;
}

.subsection-title {
    font-size: 15px;
    font-weight: 700;
    color: #0f172a;
    margin: 0 0 8px 0;
}

.kpi-card {
    padding: 10px 14px;
    border-radius: 14px;
    color: white;
    min-height: 84px;
    box-shadow: 0 6px 16px rgba(15, 23, 42, 0.12);
}

.kpi-green {
    background: linear-gradient(135deg, #28544d 0%, #1e7565 100%);
}

.kpi-orange {
    background: linear-gradient(135deg, #92400e 0%, #f59e0b 100%);
}

.kpi-red {
    background: linear-gradient(135deg, #7f1d1d 0%, #ef4444 100%);
}

.kpi-title {
    font-size: 13px;
    font-weight: 700;
    opacity: 0.85;
    margin-bottom: 5px;
}

.kpi-value {
    font-size: 19px;
    font-weight: 700;
    margin-bottom: 5px;
}

.kpi-status {
    display: inline-block;
    background: rgba(255,255,255,0.22);
    padding: 3px 10px;
    border-radius: 999px;
    font-size: 10.5px;
    font-weight: 700;
}

.brand-block {
    display: flex;
    align-items: center;
    gap: 14px;
}

.brand-mark {
    width: 50px;
    height: 50px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 24px;
    border-radius: 13px;
    background: linear-gradient(135deg, #28544d 0%, #1e7565 100%);
    color: #ffffff;
    box-shadow: 0 6px 16px rgba(30, 117, 101, 0.28);
    flex-shrink: 0;
}

.brand-title {
    font-size: 28px;
    font-weight: 700;
    letter-spacing: 0.3px;
    color: #0f172a;
    line-height: 1.2;
}

.brand-sub {
    font-size: 12.5px;
    color: #64748b;
    font-weight: 600;
    margin-top: 2px;
}

.section-title {
    font-size: 18px;
    font-weight: 700;
    color: #0f172a;
    margin: 2px 0 10px 0;
}

.clock-badge {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: #f0fdf4;
    color: #15803d;
    border: 1px solid #bbf7d0;
    border-radius: 999px;
    padding: 7px 14px;
    font-size: 14px;
    font-weight: 400;
    white-space: nowrap;
}

div[class*="st-key-chat_widget"] {
    position: fixed !important;
    bottom: 28px;
    right: 28px;
    z-index: 9999;
    width: 380px;
    max-width: 92vw;
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 18px;
    box-shadow: 0 14px 36px rgba(15, 23, 42, 0.2);
    padding: 14px;
}

div[class*="st-key-chat_toggle"] {
    position: fixed !important;
    bottom: 28px;
    right: 28px;
    z-index: 9999;
    width: 60px !important;
    height: 60px !important;
    display: flex !important;
    align-items: center;
    justify-content: center;
    overflow: visible;
}

div[class*="st-key-chat_toggle"] > div {
    width: 60px;
    height: 60px;
    display: flex;
    align-items: center;
    justify-content: center;
}

div[class*="st-key-chat_toggle"] button {
    border-radius: 999px !important;
    width: 58px !important;
    height: 58px !important;
    min-width: 58px !important;
    font-size: 22px !important;
    margin: 0 !important;
    box-shadow: 0 10px 26px rgba(15, 23, 42, 0.28);
}

.alert-red {
    background:#fff5f5;
    border-left:6px solid #ef4444;
    padding:14px 16px;
    border-radius:12px;
    margin-bottom:10px;
}

.alert-yellow {
    background:#fffaf0;
    border-left:6px solid #f59e0b;
    padding:14px 16px;
    border-radius:12px;
    margin-bottom:10px;
}

.alert-ok {
    background:#f0fdf4;
    border-left:6px solid #22c55e;
    padding:14px 16px;
    border-radius:12px;
    margin-bottom:10px;
}

.alert-detail {
    margin-top: 6px;
    font-size: 13.5px;
    color: #475569;
    line-height: 1.5;
}

.alert-detail ul {
    margin: 4px 0 0 18px;
    padding: 0;
}

.result-box {
    background:#f8fafc;
    border:1px solid #e5e7eb;
    padding:16px;
    border-radius:12px;
    margin-bottom:14px;
}

[data-testid="stCaptionContainer"],
[data-testid="stCaptionContainer"] p {
    color: #475569 !important;
    font-size: 13px !important;
    font-weight: 500 !important;
}

[data-testid="stMetricLabel"],
[data-testid="stMetricLabel"] p {
    color: #0f172a !important;
    font-size: 14px !important;
    font-weight: 700 !important;
}

[data-testid="stMetricValue"] {
    color: #0f172a !important;
    font-weight: 700 !important;
}

[data-testid="stExpander"] summary,
[data-testid="stExpander"] summary p {
    color: #0f172a !important;
    font-size: 15px !important;
    font-weight: 700 !important;
}

.chat-window {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 18px;
    padding: 18px 16px 6px 16px;
    max-height: 480px;
    overflow-y: auto;
}

.chat-row {
    display: flex;
    margin-bottom: 10px;
}

.chat-row.user {
    justify-content: flex-end;
}

.chat-row.bot {
    justify-content: flex-start;
}

.chat-bubble {
    max-width: 72%;
    padding: 10px 16px;
    border-radius: 20px;
    font-size: 14.5px;
    line-height: 1.5;
    box-shadow: 0 2px 6px rgba(15, 23, 42, 0.08);
    white-space: pre-wrap;
    word-break: break-word;
}

.chat-bubble.user {
    background: linear-gradient(135deg, #28544d 0%, #1e7565 100%);
    color: #ffffff;
    border-bottom-right-radius: 6px;
}

.chat-bubble.bot {
    background: #ffffff;
    color: #334155;
    border: 1px solid #e5e7eb;
    border-bottom-left-radius: 6px;
}
</style>
""", unsafe_allow_html=True)


def find_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


@st.cache_resource
def load_resources():
    model_path = find_path(MODEL_CANDIDATES)
    if model_path is None:
        raise FileNotFoundError("best_LightGBM_mass_speed_regressor.pkl 파일을 찾을 수 없습니다.")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    df = pd.read_csv(CSV_PATH, encoding="cp949")

    paper_rules = []
    if PAPER_JSON_PATH.exists():
        with open(PAPER_JSON_PATH, "r", encoding="utf-8") as f:
            paper_rules = json.load(f)

    opls_rules = []
    if OPLS_JSON_PATH.exists():
        with open(OPLS_JSON_PATH, "r", encoding="utf-8") as f:
            opls_rules = json.load(f)

    shap_path = find_path(SHAP_CANDIDATES)
    shap_text = shap_path.read_text(encoding="utf-8") if shap_path else ""

    return model, df, paper_rules, opls_rules, shap_text, model_path


model, df_mes, paper_rules, opls_rules, shap_text, model_path = load_resources()

E5_PREFIX = "Instruct: 공정 이상 원인과 조치 방법을 찾으세요\nQuery: "

CHAT_RAG_TEMPLATE = """다음 문맥을 참고하여 질문에 답변해 주세요.

문맥에는 논문 기반 공정 rule, OPLS 공정 기준, SHAP 기반 모델 해석 정보가 포함됩니다.

[지식베이스 컨텍스트]
{knowledge_context}

답변 시 아래 내용을 중심으로 정리해 주세요.
- 핵심 답변
- 관련 공정 변수
- 모델/문헌 기반 근거
- 조치 방향

질문:
{question}

답변:
"""


@st.cache_resource(show_spinner="PRAGma 모델 로딩 중... (최초 1회, 수 분 소요)")
def load_chat_pipeline():
    if USE_GEMINI:
        emb_model = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=GEMINI_API_KEY,
        )
    else:
        if not torch.cuda.is_available():
            return None, None

        emb_model = HuggingFaceEmbeddings(
            model_name="intfloat/multilingual-e5-large-instruct",
            encode_kwargs={"normalize_embeddings": True},
        )

    docs = []
    for item in paper_rules:
        docs.append(Document(
            page_content=json.dumps(item, ensure_ascii=False, indent=2),
            metadata={"type": "paper_rule"},
        ))
    for item in opls_rules:
        docs.append(Document(
            page_content=json.dumps(item, ensure_ascii=False, indent=2),
            metadata={"type": "opls_rule"},
        ))
    if shap_text:
        docs.append(Document(page_content=shap_text, metadata={"type": "shap_analysis"}))

    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
    split_docs = splitter.split_documents(docs)

    chroma_suffix = "gemini" if USE_GEMINI else "exaone"
    database = Chroma.from_documents(
        documents=split_docs,
        embedding=emb_model,
        collection_name=f"pragma_rag_{chroma_suffix}",
        persist_directory=str(BASE_DIR / f"pragma_chroma_{chroma_suffix}"),
    )
    vector_ret = database.as_retriever(search_kwargs={"k": 5})
    bm25_ret = BM25Retriever.from_documents(split_docs)
    bm25_ret.k = 5

    class _HybridRetriever(BaseRetriever):
        def _get_relevant_documents(self, query, *, run_manager: CallbackManagerForRetrieverRun = None):
            vec_docs = vector_ret.invoke(query if USE_GEMINI else E5_PREFIX + query)
            bm25_docs = bm25_ret.invoke(query)
            seen, combined = set(), []
            for doc in vec_docs + bm25_docs:
                key = doc.page_content[:80]
                if key not in seen:
                    seen.add(key)
                    combined.append(doc)
            return combined[:7]

        async def _aget_relevant_documents(self, query, **kwargs):
            return self._get_relevant_documents(query)

    retriever = _HybridRetriever()

    if USE_GEMINI:
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            google_api_key=GEMINI_API_KEY,
            temperature=0.3,
        )
    else:
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        pipeline = HuggingFacePipeline.from_model_id(
            model_id="LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
            task="text-generation",
            pipeline_kwargs={"max_new_tokens": 512, "do_sample": False, "repetition_penalty": 1.03},
            model_kwargs={"quantization_config": quant_cfg, "trust_remote_code": True},
        )
        llm = ChatHuggingFace(llm=pipeline)

    return retriever, llm


rag_retriever, chat_llm = load_chat_pipeline()

try:
    MODEL_FEATURES = model.feature_name_
except Exception:
    MODEL_FEATURES = model.booster_.feature_name()


@st.cache_resource
def load_shap_explainer():
    return shap.TreeExplainer(model)


shap_explainer = load_shap_explainer()


def explain_prediction(X, top_n=10):
    """예측에 대한 LOT별 SHAP 기여도 상위 N개 피처를 반환"""
    shap_values = shap_explainer.shap_values(X)
    values = shap_values[0] if isinstance(shap_values, list) else shap_values[0]
    contrib = pd.Series(values, index=MODEL_FEATURES)
    return contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(top_n)


CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val": "cu_thick_max",
    "Cu 표면두께 AVG_VAL": "cu_thick_avg",
    "Cu 표면두께 Min_Val": "cu_thick_min",
    "Cu 표면두께 Std_Val": "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}

SUFFIX_MAP = {
    "Etch factor": "etch_factor",
    "Etching(염화동) - Cu": "meas_etch_cu",
    "Etching(염화동) - HCl": "meas_etch_hcl",
    "Etching(염화동) - 비중": "meas_etch_sg",
    "Etching(염화동) - 온도": "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량": "meas_etch_amount",
    "Soft Etch - Cu": "meas_softetch_cu",
    "Soft Etch - H2SO4": "meas_softetch_h2so4",
    "Soft Etch - SPS": "meas_softetch_sps",
    "박리액 - 농도": "meas_strip_conc",
    "수세수 - pH": "meas_rinse_ph",
    "현상액 - pH": "meas_dev_ph",
    "현상액 - 농도": "meas_dev_conc",
}

for col in df_mes.columns:
    if "분석치" in col or "分" in col:
        suffix = col.split("_", 1)[-1] if "_" in col else col
        if suffix in SUFFIX_MAP:
            CONTINUOUS_MAP[col] = SUFFIX_MAP[suffix]


CATEGORICAL_COLS = {
    "재작업사유": {
        "prefix": "rework_history",
        "values": ["Unknown", "기타", "기판 겹침", "두께 미달", "딤플", "설비 에러"],
        "sep": "_",
    },
    "노광 설비정보": {
        "prefix": "expo_eq_id",
        "values": [f"EXP-{i:03d}" for i in range(1, 8)],
        "sep": "_",
    },
    "DES 설비정보": {
        "prefix": "des_eq_id",
        "values": [f"DES-{i:03d}" for i in range(1, 7)],
        "sep": "_",
    },
    "정면 설비정보": {
        "prefix": "brush_eq_id",
        "values": [f"PRE-{i:03d}" for i in range(1, 8)],
        "sep": "_",
    },
}


OPLS_BOUNDS = {
    "meas_etch_temp": {"name": "에칭 온도", "lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C"},
    "meas_etch_sg": {"name": "에칭 비중", "lcl": 1.32, "sl": 1.37, "ucl": 1.42, "unit": ""},
    "meas_etch_cu": {"name": "에칭 Cu 농도", "lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L"},
    "meas_etch_hcl": {"name": "에칭 HCl", "lcl": 0.3, "sl": 0.5, "ucl": 0.7, "unit": "N"},
    "meas_etch_additive": {"name": "에칭 첨가제", "lcl": 2.6, "sl": 3.0, "ucl": 3.4, "unit": "g/L"},
}


def get_lot_data(lot_no):
    rows = df_mes[df_mes["LOT"] == lot_no]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()


def prepare_features(lot_data):
    row = {}

    for csv_col, feat in CONTINUOUS_MAP.items():
        val = lot_data.get(csv_col, np.nan)
        row[feat] = float(val) if val is not None and str(val) not in ["", "nan", "None"] else np.nan

    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw = str(lot_data.get(csv_col, "")).strip()
        norm = raw.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw

        for value in cfg["values"]:
            norm_value = value.replace(" ", "_") if cfg["prefix"] == "rework_history" else value
            key = f"{cfg['prefix']}{cfg['sep']}{norm_value}"
            row[key] = 1 if norm == norm_value else 0

    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0

    return pd.DataFrame([row])[MODEL_FEATURES]


def predict_speed(lot_data):
    X = prepare_features(lot_data)
    pred = float(model.predict(X)[0])
    return pred, X


def check_opls(X):
    result = []

    for feat, bound in OPLS_BOUNDS.items():
        if feat not in X.columns:
            continue

        value = X[feat].iloc[0]
        if pd.isna(value):
            continue

        status = "정상"
        level = "ok"

        if value > bound["ucl"]:
            status = "UCL 초과"
            level = "high"
        elif value < bound["lcl"]:
            status = "LCL 미달"
            level = "high"
        elif value > bound["sl"] * 1.02:
            status = "SL 상향 근접"
            level = "mid"
        elif value < bound["sl"] * 0.98:
            status = "SL 하향 근접"
            level = "mid"

        result.append({
            "피처": feat,
            "항목": bound["name"],
            "현재값": round(float(value), 4),
            "LCL": bound["lcl"],
            "SL": bound["sl"],
            "UCL": bound["ucl"],
            "단위": bound["unit"],
            "상태": status,
            "위험도": level,
        })

    return result


def chatbot_answer(question):
    if not question or not question.strip():
        return "질문을 입력해 주세요."

    if rag_retriever is None or chat_llm is None:
        if USE_GEMINI:
            return ("⚠️ Gemini API 키가 설정되지 않아 PRAGma 모델을 불러오지 못했습니다. "
                    "Colab Secrets에 GEMINI_API_KEY를 등록한 뒤 다시 실행해 주세요.")
        return ("⚠️ GPU를 사용할 수 없어 PRAGma 모델을 불러오지 못했습니다. "
                "Colab 런타임 유형을 GPU로 변경한 뒤 다시 실행해 주세요.")

    docs = rag_retriever.invoke(question)
    knowledge_context = "\n\n".join(
        f"[type={d.metadata.get('type', '')}]\n{d.page_content}" for d in docs
    )

    prompt_text = CHAT_RAG_TEMPLATE.format(
        knowledge_context=knowledge_context,
        question=question,
    )

    return chat_llm.invoke(prompt_text).content


DES_EQUIPMENT_LIST = CATEGORICAL_COLS["DES 설비정보"]["values"]

METRIC_BOUNDS = {
    "에칭 온도": OPLS_BOUNDS["meas_etch_temp"],
    "에칭 비중": OPLS_BOUNDS["meas_etch_sg"],
    "에칭 Cu 농도": OPLS_BOUNDS["meas_etch_cu"],
    "에칭 첨가제": OPLS_BOUNDS["meas_etch_additive"],
}

MONITOR_UNITS = {"에칭 온도": "°C", "에칭 비중": "", "에칭 Cu 농도": "g/L", "에칭 첨가제": "g/L"}
MONITOR_DIGITS = {"에칭 온도": 2, "에칭 비중": 3, "에칭 Cu 농도": 1, "에칭 첨가제": 2}
MONITOR_WINDOW = 30

MONITOR_ACTIONS = {
    "에칭 온도": ["칠러 설정 온도 확인", "스프레이 압력/유량 확인"],
    "에칭 비중": ["보메(비중) 트렌드 확인", "신액/배액 밸런스 확인"],
    "에칭 Cu 농도": ["Auto Drain 확인", "신액 보충 진행", "Etch factor 동시 점검"],
    "에칭 첨가제": ["첨가제 토출량 확인", "펌프 에어록 제거", "노즐 상태 점검"],
}


def _metric_status(value, bound):
    if value > bound["ucl"]:
        return "UCL 초과", "high", "kpi-red", "alert-red"
    if value < bound["lcl"]:
        return "LCL 미달", "high", "kpi-red", "alert-red"
    if value > bound["sl"] * 1.02 or value < bound["sl"] * 0.98:
        return "주의", "mid", "kpi-orange", "alert-yellow"
    return "정상", "ok", "kpi-green", None


def _walk_step(state, bound, rng):
    """평소엔 SL 부근에서 완만하게 흐르다가, 가끔 한쪽으로 튀어 UCL/LCL을 벗어난 뒤
    서서히 정상 범위로 복귀하는 실제 공정 트렌드를 흉내낸다."""
    sl = bound["sl"]
    span = bound["ucl"] - bound["lcl"]
    value = state["value"]

    if state["phase"] == "normal":
        value += (sl - value) * 0.10 + rng.normal(0, span * 0.006)
        if rng.random() < 0.012:
            direction = rng.choice([-1, 1])
            state["target"] = sl + direction * span * rng.uniform(0.8, 1.1)
            state["phase"] = "spike"
            state["ticks"] = int(rng.integers(2, 4))
    elif state["phase"] == "spike":
        value += (state["target"] - value) * 0.55 + rng.normal(0, span * 0.006)
        state["ticks"] -= 1
        if state["ticks"] <= 0:
            state["phase"] = "recover"
            state["ticks"] = int(rng.integers(4, 7))
    else:
        value += (sl - value) * 0.30 + rng.normal(0, span * 0.006)
        state["ticks"] -= 1
        if state["ticks"] <= 0:
            state["phase"] = "normal"

    state["value"] = float(np.clip(value, bound["lcl"] - span * 0.2, bound["ucl"] + span * 0.2))
    return state["value"]


def _init_monitor_state(eq_id):
    seed = int.from_bytes(eq_id.encode("utf-8"), "little") % (2**32)
    return {
        "rng": np.random.default_rng(seed),
        "buffer": {
            "시간": [f"T-{(MONITOR_WINDOW - i) * 10}s" for i in range(MONITOR_WINDOW)],
            **{name: [bound["sl"]] * MONITOR_WINDOW for name, bound in METRIC_BOUNDS.items()},
        },
        "walk": {
            name: {"phase": "normal", "ticks": 0, "value": bound["sl"]}
            for name, bound in METRIC_BOUNDS.items()
        },
    }


def _advance_monitor_state(eq_id):
    monitor_states = st.session_state.setdefault("monitor_states", {})
    state = monitor_states.setdefault(eq_id, _init_monitor_state(eq_id))
    rng, buf = state["rng"], state["buffer"]

    buf["시간"].append(datetime.now().strftime("%H:%M:%S"))
    buf["시간"].pop(0)

    for name, bound in METRIC_BOUNDS.items():
        value = _walk_step(state["walk"][name], bound, rng)
        buf[name].append(value)
        buf[name].pop(0)

    return pd.DataFrame(buf)


def make_deviation_figure(df, col, bound):
    sl = bound["sl"]
    ucl_dev = (bound["ucl"] - sl) / sl * 100
    lcl_dev = (bound["lcl"] - sl) / sl * 100
    deviation = ((df[col] - sl) / sl) * 100
    fig = go.Figure(go.Scatter(
        x=df["시간"],
        y=deviation,
        mode="lines",
        line=dict(color="#1e7565", width=2),
        fill="tozeroy",
        fillcolor="rgba(30, 117, 101, 0.12)",
    ))
    fig.add_hline(y=0, line_color="#cbd5e1", line_width=1)
    fig.add_hline(
        y=ucl_dev, line_color="#ef4444", line_width=1.5, line_dash="dash",
        annotation_text="UCL", annotation_position="top right",
        annotation_font_color="#ef4444", annotation_font_size=11,
    )
    fig.add_hline(
        y=lcl_dev, line_color="#ef4444", line_width=1.5, line_dash="dash",
        annotation_text="LCL", annotation_position="bottom right",
        annotation_font_color="#ef4444", annotation_font_size=11,
    )
    fig.update_layout(
        height=120,
        margin=dict(l=6, r=6, t=6, b=6),
        paper_bgcolor="#f8fafc",
        plot_bgcolor="#f8fafc",
        xaxis=dict(showgrid=False, showticklabels=False),
        yaxis=dict(showticklabels=False, gridcolor="#e5e7eb", zeroline=False),
        showlegend=False,
    )
    return fig


def render_kpi_card(title, value, unit, status, color):
    st.markdown(
        f"""
        <div class="kpi-card {color}">
            <div class="kpi-title">{title}</div>
            <div class="kpi-value">{value}<span style="font-size:18px;"> {unit}</span></div>
            <div class="kpi-status">{status}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


@st.fragment(run_every="1s")
def render_clock():
    now = datetime.now()
    weekday = ["월", "화", "수", "목", "금", "토", "일"][now.weekday()]
    st.markdown(
        f'<div class="clock-badge">● 실시간 · {now.strftime("%Y-%m-%d")}({weekday}) {now.strftime("%H:%M:%S")}</div>',
        unsafe_allow_html=True,
    )


@st.fragment(run_every="10s")
def render_monitoring():
    selected_eq = st.selectbox("모니터링 설비 선택", DES_EQUIPMENT_LIST, key="monitor_eq_select")
    st.session_state["monitor_selected_eq"] = selected_eq

    df = _advance_monitor_state(selected_eq)
    names = list(METRIC_BOUNDS.keys())
    alerts = []

    kpi_cols = st.columns(4)
    chart_cols = st.columns(4)

    for kpi_cell, chart_cell, name in zip(kpi_cols, chart_cols, names):
        bound = METRIC_BOUNDS[name]
        value = df[name].iloc[-1]
        status, level, color, alert_class = _metric_status(value, bound)
        with kpi_cell:
            render_kpi_card(name, f"{value:.{MONITOR_DIGITS[name]}f}", MONITOR_UNITS[name], status, color)
        with chart_cell:
            st.plotly_chart(make_deviation_figure(df, name, bound), use_container_width=True)
        if level != "ok":
            alerts.append((name, bound, value, status, level, alert_class))

    now_str = datetime.now().strftime("%H:%M:%S")
    alarm_logs = st.session_state.setdefault("process_alarm_logs", {})
    alarm_log = alarm_logs.setdefault(selected_eq, {})
    active_names = {a[0] for a in alerts}
    for name in active_names:
        alarm_log.setdefault(name, now_str)
    for name in list(alarm_log.keys()):
        if name not in active_names:
            del alarm_log[name]

    alerts_map = st.session_state.setdefault("process_alerts_map", {})
    alerts_map[selected_eq] = alerts


@st.fragment(run_every="10s")
def render_alarm_popover():
    selected_eq = st.session_state.get("monitor_selected_eq", DES_EQUIPMENT_LIST[0])
    alerts = st.session_state.get("process_alerts_map", {}).get(selected_eq, [])
    alarm_log = st.session_state.get("process_alarm_logs", {}).get(selected_eq, {})
    high = [a for a in alerts if a[4] == "high"]
    mid = [a for a in alerts if a[4] == "mid"]

    if high:
        badge_label = f"🔴 {selected_eq} 알람 {len(alerts)}"
    elif mid:
        badge_label = f"🟠 {selected_eq} 알람 {len(alerts)}"
    else:
        badge_label = f"🟢 {selected_eq} 정상"

    with st.popover(badge_label, use_container_width=True):
        st.markdown(f'<div class="subsection-title">공정 알람 — {selected_eq}</div>', unsafe_allow_html=True)

        if not alerts:
            st.markdown(
                """
                <div class="alert-ok">
                <b>현재 이탈 항목이 없습니다</b>
                <div class="alert-detail">모든 공정 변수가 관리 기준(LCL~UCL) 내에서 운영되고 있습니다.</div>
                </div>
                """,
                unsafe_allow_html=True,
            )
        else:
            for name, bound, value, status, level, alert_class in alerts:
                occurred_at = alarm_log.get(name, datetime.now().strftime("%H:%M:%S"))
                action_html = "".join(f"<li>{a}</li>" for a in MONITOR_ACTIONS.get(name, []))
                st.markdown(
                    f"""
                    <div class="{alert_class}">
                    <b>{occurred_at}에 {name} {status} 발생</b>
                    <div class="alert-detail">
                    현재값 {value:.3f}{MONITOR_UNITS[name]}
                    (LCL {bound['lcl']} / SL {bound['sl']} / UCL {bound['ucl']})
                    </div>
                    <div class="alert-detail"><b>권장 조치</b><ul>{action_html}</ul></div>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )


def render_monitoring_section():
    st.markdown('<div class="section-title">📊 실시간 모니터링</div>', unsafe_allow_html=True)
    render_monitoring()


def render_speed_calculator():
    st.markdown('<div class="section-title">🏭 DES 컨베이어 속도 산출</div>', unsafe_allow_html=True)

    input_col, button_col = st.columns([3, 1], vertical_alignment="bottom")
    with input_col:
        lot_no = st.text_input("LOT 번호", value="A20000", key="lot_no_input")
    with button_col:
        run = st.button("산출 실행", type="primary", use_container_width=True)

    lot_preview = get_lot_data(lot_no)

    if lot_preview is not None:
        st.markdown(
            '<div style="font-size:14px; font-weight:700; color:#0f172a; margin:0 0 8px 0;">LOT 정보</div>',
            unsafe_allow_html=True,
        )
        with st.container(border=True, key="lot_info_box"):
            info_col1, info_col2, info_col3, info_col4 = st.columns(4)

            with info_col1:
                st.text_input("제품군", value=str(lot_preview.get("제품군", "N/A")), disabled=True)
                st.text_input("거래처", value=str(lot_preview.get("거래처", "N/A")), disabled=True)

            with info_col2:
                st.text_input("LAYER", value=str(lot_preview.get("LAYER", "N/A")), disabled=True)
                st.text_input("공법구분", value=str(lot_preview.get("공법구분", "N/A")), disabled=True)

            with info_col3:
                st.text_input("도금구분", value=str(lot_preview.get("도금구분", "N/A")), disabled=True)
                st.text_input("DRY FILM 정보", value=str(lot_preview.get("DRY FILM 정보", "N/A")), disabled=True)

            with info_col4:
                st.text_input("노광 설비정보", value=str(lot_preview.get("노광 설비정보", "N/A")), disabled=True)
                st.text_input("DES 설비정보", value=str(lot_preview.get("DES 설비정보", "N/A")), disabled=True)

    if run:
        st.session_state["speed_calc_lot"] = lot_no

    if st.session_state.get("speed_calc_lot") != lot_no:
        return

    lot = get_lot_data(lot_no)

    if lot is None:
        st.error(f"'{lot_no}' LOT를 찾을 수 없습니다.")
        sample_lots = ", ".join(df_mes["LOT"].astype(str).head(10).tolist())
        st.caption(f"사용 가능한 LOT 예시: {sample_lots}")
        return

    pred, X = predict_speed(lot)
    alerts = check_opls(X)

    metric_col, decision_col = st.columns([2, 3])

    with metric_col:
        st.markdown(
            f"""
            <div class="result-box" style="display:flex; flex-direction:column; align-items:center; justify-content:center; text-align:center; height:100%;">
                <div style="font-size:14px; font-weight:700; color:#0f172a; margin-bottom:6px;">예측 DES 컨베이어 속도</div>
                <div style="font-size:28px; font-weight:700; color:#0f172a;">{round(pred, 2):.2f}<span style="font-size:15px; font-weight:600; color:#475569;"> m/min</span></div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    with decision_col:
        approve_col, reject_col = st.columns(2)
        with approve_col:
            approve_clicked = st.button("승인", key="speed_approve_btn", type="primary", use_container_width=True)
        with reject_col:
            reject_clicked = st.button("반려", key="speed_reject_btn", use_container_width=True)

        reason = st.text_input(
            "사유", key="speed_decision_reason", placeholder="승인/반려 사유를 입력하세요 (선택)",
            label_visibility="collapsed",
        )

        if approve_clicked or reject_clicked:
            decision = "승인" if approve_clicked else "반려"
            decision_log = st.session_state.setdefault("speed_decision_log", [])
            decision_log.insert(0, {
                "시간": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "LOT": lot_no,
                "예측 속도(m/min)": round(pred, 2),
                "결정": decision,
                "사유": reason if reason.strip() else "-",
            })
            st.success(f"'{lot_no}' LOT 예측 속도 {decision} 처리되었습니다.")

    decision_log = st.session_state.get("speed_decision_log", [])
    if decision_log:
        st.markdown('<div class="subsection-title">승인/반려 이력</div>', unsafe_allow_html=True)
        st.dataframe(pd.DataFrame(decision_log), use_container_width=True, hide_index=True)

    st.dataframe(pd.DataFrame(alerts).drop(columns=["피처"]), use_container_width=True)

    st.markdown('<div class="subsection-title">예측 근거 (SHAP 기여 분석)</div>', unsafe_allow_html=True)
    st.caption("SHAP 값을 기준으로 각 공정 변수가 예측 속도에 미친 영향의 방향(+/−)과 크기를 정량적으로 분석한 결과입니다.")

    top_shap = explain_prediction(X, top_n=10)
    shap_colors = ["#E74C3C" if v > 0 else "#3498DB" for v in top_shap.values]

    fig_shap = go.Figure(go.Bar(
        x=top_shap.values,
        y=top_shap.index,
        orientation="h",
        marker_color=shap_colors,
    ))
    fig_shap.update_layout(
        paper_bgcolor="#f8fafc",
        plot_bgcolor="#f8fafc",
        xaxis_title="SHAP value (← 속도 ↓  |  속도 ↑ →)",
        yaxis=dict(autorange="reversed"),
        height=320,
        margin=dict(l=10, r=10, t=30, b=10),
    )
    fig_shap.add_vline(x=0, line_color="#888", line_width=1)
    st.plotly_chart(fig_shap, use_container_width=True)


title_col, clock_col, alarm_col = st.columns([3, 1, 1])

with title_col:
    st.markdown(
        """
        <div class="brand-block">
            <div class="brand-mark">⚙️</div>
            <div>
                <div class="brand-title">PRAGma</div>
                <div class="brand-sub">DES Process AI Agentic Platform</div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

with clock_col:
    st.write("")
    render_clock()

with alarm_col:
    st.write("")
    render_alarm_popover()

render_monitoring_section()

st.divider()

render_speed_calculator()

st.divider()

DEFAULT_CHAT_GREETING = {"role": "bot", "content": "안녕하세요! 에칭 공정 관련 궁금한 점을 편하게 물어보세요."}

if "chat_messages" not in st.session_state:
    st.session_state["chat_messages"] = [dict(DEFAULT_CHAT_GREETING)]
if "chat_open" not in st.session_state:
    st.session_state["chat_open"] = False


def render_chat_panel():
    head_col, reset_col, close_col = st.columns([4, 1, 1])
    with head_col:
        st.markdown('<div class="subsection-title">PRAGma</div>', unsafe_allow_html=True)
    with reset_col:
        if st.button("🔄", key="chat_reset_btn", help="대화 초기화", use_container_width=True):
            st.session_state["chat_messages"] = [dict(DEFAULT_CHAT_GREETING)]
            st.rerun()
    with close_col:
        if st.button("✕", key="chat_close_btn", help="닫기", use_container_width=True):
            st.session_state["chat_open"] = False
            st.rerun()

    st.caption("예시 질문을 누르면 AI가 바로 분석해 드립니다.")

    quick_questions = [
        "Cu 농도가 UCL 초과하면 어떤 조치를 해야 하나요?",
        "과에칭 발생 시 원인과 조치 방향을 알려줘.",
        "Etch factor 저하 시 원인과 조치 방향을 알려줘.",
    ]

    quick_question = None
    for q in quick_questions:
        if st.button(q, key=f"chat_quick_{q}", use_container_width=True):
            quick_question = q

    bubbles = []
    for msg in st.session_state["chat_messages"]:
        role = "user" if msg["role"] == "user" else "bot"
        safe_text = (
            msg["content"]
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
        )
        bubbles.append(
            f'<div class="chat-row {role}"><div class="chat-bubble {role}">{safe_text}</div></div>'
        )

    st.markdown(f'<div class="chat-window">{"".join(bubbles)}</div>', unsafe_allow_html=True)

    # 채팅창을 항상 최신 메시지가 보이도록 맨 아래로 자동 스크롤
    components.html(
        """
        <script>
        const win = window.parent.document.querySelector('.chat-window');
        if (win) { win.scrollTop = win.scrollHeight; }
        </script>
        """,
        height=0,
    )

    with st.form(key="chat_form", clear_on_submit=True):
        typed_question = st.text_input(
            "질문", value="", placeholder="질문을 입력하세요...", label_visibility="collapsed",
        )
        submitted = st.form_submit_button("전송", use_container_width=True)

    new_question = quick_question or (typed_question.strip() if submitted and typed_question.strip() else None)

    if new_question:
        st.session_state["chat_messages"].append({"role": "user", "content": new_question})
        with st.spinner("PRAGma가 답변을 작성하고 있습니다..."):
            answer = chatbot_answer(new_question)
        st.session_state["chat_messages"].append({"role": "bot", "content": answer})
        st.rerun()


if st.session_state["chat_open"]:
    with st.container(key="chat_widget"):
        render_chat_panel()
else:
    with st.container(key="chat_toggle"):
        if st.button("💬", key="chat_open_btn", help="공정 지식 챗봇 열기"):
            st.session_state["chat_open"] = True
            st.rerun()
'''

Path("/content/PRAGma/pragma_streamlit.py").write_text(APP_CODE, encoding="utf-8")
print("완료: /content/PRAGma/pragma_streamlit.py 생성됨")

완료: /content/PRAGma/pragma_streamlit.py 생성됨


## 7. Streamlit 서버 실행

In [110]:
import subprocess
import time

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/PRAGma/pragma_streamlit.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)

print("Streamlit 서버 실행 완료")

Streamlit 서버 실행 완료


## 8. Cloudflare Tunnel 실행

In [111]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [112]:
import subprocess
import re

cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(80):
    line = cf.stdout.readline()
    print(line, end="")

    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)

    if match:
        print("\n앱 주소:", match.group(0))
        break

2026-06-08T15:28:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-08T15:28:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-08T15:28:07Z INF +--------------------------------------------------------------------------------------------+
2026-06-08T15:28:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-08T15:28:07Z INF |  https://rat-bailey-modular-gap.trycloudflare.com     

## 9. 세션이 끊겼을 때 재실행

Streamlit/Cloudflare 프로세스를 종료 후 재시작하고 새 터널 주소를 발급합니다.

In [113]:
import subprocess
import time
import re

# 기존 프로세스 종료
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

# Streamlit 실행
streamlit_proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/PRAGma/pragma_streamlit.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)

# Cloudflare 터널 실행
cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(80):
    line = cf.stdout.readline()
    print(line, end="")

    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)

    if match:
        print("\n새 앱 주소:", match.group(0))
        break

2026-06-08T15:28:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-08T15:28:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-08T15:28:16Z INF +--------------------------------------------------------------------------------------------+
2026-06-08T15:28:16Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-08T15:28:16Z INF |  https://documented-worcester-predict-blackberry.trycl